[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tasmia008/Complete_Guidelines_LLM_FineTuning/blob/main/13_more_alignment.ipynb)

# More alignment — ORPO, KTO, reward modeling

> **Fine-tuning series — 13 of 26.** A standalone notebook in a set; see `00_README.md` for the full index and order. This notebook is self-contained: run **Setup**, then the rest.


## Setup (run first)

Self-contained: imports, model id, device flags, and a tiny inline dataset so this notebook runs on its own.

In [ ]:
# Environment check — record these versions with any results you report.
import importlib
for lib in ["torch", "transformers", "peft", "trl", "datasets",
            "bitsandbytes", "accelerate", "unsloth", "adapters"]:
    try:
        m = importlib.import_module(lib)
        print(f"{lib:14s} {getattr(m, '__version__', '?')}")
    except Exception as e:
        print(f"{lib:14s} not installed ({type(e).__name__})")

In [ ]:
# !pip install -U transformers peft trl bitsandbytes datasets accelerate
# !pip install unsloth   # only for the Unsloth section (CUDA only)
import gc, torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"     # small + fast; swap for any causal LM
device = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")
BF16_OK = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
FP16_OK = torch.cuda.is_available() and not BF16_OK   # fp16 needs a GPU
print("device:", device, "| bf16:", BF16_OK)

def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# (a) Instruction data -> for SFT / LoRA / QLoRA / Unsloth / instruction / prompt tuning
instructions = [
    {"instruction": "Give the capital of France.", "output": "Paris."},
    {"instruction": "What is 7 times 8?", "output": "56."},
    {"instruction": "Translate 'good morning' to Spanish.", "output": "Buenos días."},
    {"instruction": "Name the largest planet.", "output": "Jupiter."},
    {"instruction": "Define photosynthesis in one line.",
     "output": "Plants converting sunlight, water, and CO2 into energy and oxygen."},
] * 8   # repeat just to have enough steps in the demo

# (b) Preference data -> for DPO (prompt + chosen/rejected, no reward model)
preferences = [
    {"prompt": "Explain gravity to a child.",
     "chosen": "Gravity is the pull that brings things down to the ground.",
     "rejected": "Gravity is a tensor field obeying the Einstein equations."},
    {"prompt": "Recommend a book.",
     "chosen": "Try 'The Hobbit' — a fun, easy adventure.",
     "rejected": "Read whatever, I don't care."},
] * 16

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def to_chat_text(ex):
    msgs = [{"role": "user", "content": ex["instruction"]},
            {"role": "assistant", "content": ex["output"]}]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False)}

sft_ds = Dataset.from_list([to_chat_text(e) for e in instructions])
pref_ds = Dataset.from_list(preferences)
print(sft_ds[0]["text"][:160])

In [ ]:
# Shared trainer imports + a default LoRA config reused by several sections.
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig
lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"])

## 9. More alignment (what-axis): ORPO · KTO · reward modeling

Beyond DPO. **ORPO** folds SFT *and* preference into one loss with **no reference
model and no separate SFT pass**. **KTO** aligns from **unpaired** binary feedback
(thumbs up/down) — no preference pairs needed. **Reward modeling** trains the scalar
reward model that PPO / best-of-N rely on.

> TRL v1 moved several preference trainers into a `trl.experimental` namespace, so the
> imports below try the stable path first and fall back.

**ORPO** — reference-free; preference data; stack on LoRA via `peft_config`.

In [ ]:
try:
    from trl import ORPOConfig, ORPOTrainer
except ImportError:
    from trl.experimental.orpo import ORPOConfig, ORPOTrainer

orpo_trainer = ORPOTrainer(
    model=MODEL_ID,
    args=ORPOConfig(output_dir="ft_orpo", per_device_train_batch_size=1,
        gradient_accumulation_steps=4, max_steps=20, learning_rate=5e-6, beta=0.1,
        max_length=512, bf16=BF16_OK, fp16=FP16_OK,   # max_prompt_length removed in TRL >= 1.0
        logging_steps=5, report_to="none"),
    train_dataset=pref_ds, peft_config=lora_cfg)
orpo_trainer.train(); del orpo_trainer; cleanup()

**KTO** — unpaired `{prompt, completion, label(bool)}`; built here from our pairs.

In [ ]:
try:
    from trl import KTOConfig, KTOTrainer
except ImportError:
    from trl.experimental.kto import KTOConfig, KTOTrainer

kto_ds = Dataset.from_list(
    [{"prompt": p["prompt"], "completion": p["chosen"],   "label": True}  for p in preferences] +
    [{"prompt": p["prompt"], "completion": p["rejected"], "label": False} for p in preferences])

kto_trainer = KTOTrainer(
    model=MODEL_ID,
    args=KTOConfig(output_dir="ft_kto", per_device_train_batch_size=2, max_steps=20,
        learning_rate=5e-6, beta=0.1, max_length=512,   # max_prompt_length removed in TRL >= 1.0
        bf16=BF16_OK, fp16=FP16_OK, logging_steps=5, report_to="none"),
    train_dataset=kto_ds, peft_config=lora_cfg)
kto_trainer.train(); del kto_trainer; cleanup()

**Reward modeling** — a 1-output classifier scored so chosen > rejected.

In [ ]:
from trl import RewardTrainer, RewardConfig
from transformers import AutoModelForSequenceClassification

rm = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, num_labels=1)
rm.config.pad_token_id = tokenizer.pad_token_id
reward_ds = Dataset.from_list([
    {"chosen": p["prompt"] + "\n" + p["chosen"],
     "rejected": p["prompt"] + "\n" + p["rejected"]} for p in preferences])

reward_trainer = RewardTrainer(
    model=rm,
    args=RewardConfig(output_dir="ft_reward", per_device_train_batch_size=2,
        max_steps=20, learning_rate=1e-5, max_length=512, bf16=BF16_OK,
        logging_steps=5, report_to="none"),
    train_dataset=reward_ds, processing_class=tokenizer)
reward_trainer.train(); del reward_trainer, rm; cleanup()

**SimPO** — like DPO but **reference-free**: it drops the frozen reference model and instead
uses the *length-normalized average log-probability* as the implicit reward, plus a target margin
that pushes chosen and rejected apart. Fewer moving parts (no reference model in memory) and strong
results. In TRL it's implemented via the `CPOTrainer` with `CPOConfig(loss_type="simpo",
cpo_alpha=...)` on standard `{prompt, chosen, rejected}` data. As of TRL ≥ 1.0 CPO lives in the
experimental namespace: `from trl.experimental.cpo import CPOConfig, CPOTrainer`.